## MUSIC POPULARITY PROJECT

### Group 6

### Project Overview
This project is carried out by the data team at a music streaming service. The A&R (Artists & Repertoire) department wants to use data to make better decisions about which music to back, so we analyse Spotify track data to answer three questions.

### Objectives
1. **What makes a track popular?** (Regression): identify which audio features and track characteristics best explain a song's popularity score.
2. **Can we spot a hit before it charts?** (Classification): build a model that predicts whether a track is likely to be a hit, using only information available before release.
3. **What moods exist inside a genre?** (Clustering): group tracks within our chosen genre into distinct mood profiles based on their audio features.

### Dataset
- **Source:** Kaggle
- **File:** `spotify_tracks.csv`

### Team

### Load the Data

In [ ]:
# Import of libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

In [ ]:
# Select the genre
GENRE = 'acoustic'      

RANDOM_STATE = 42   

print(f'Working on: {GENRE}')

Working on: acoustic


In [2]:
# Load the dataset
df = pd.read_csv('spotify_tracks.csv')

### Review Structure and Summary

In [ ]:
# Print first few rows
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
# Check the shape of the dataframe(number of rows and columns)
df.shape

(114000, 21)

In [6]:
# Check the data types of each column
df.dtypes

Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object

In [8]:
# Check for the data information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

In [9]:
# Summary statistics of the numerical columns
df.describe()

,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,114000.000000,114000.000000,1.140000e+05,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000
mean,56999.500000,33.238535,2.280292e+05,0.566800,0.641383,5.309140,-8.258960,0.637553,0.084652,0.314910,0.156050,0.213553,0.474068,122.147837,3.904035
std,32909.109681,22.305078,1.072977e+05,0.173542,0.251529,3.559987,5.029337,0.480709,0.105732,0.332523,0.309555,0.190378,0.259261,29.978197,0.432621
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,28499.750000,17.000000,1.740660e+05,0.456000,0.472000,2.000000,-10.013000,0.000000,0.035900,0.016900,0.000000,0.098000,0.260000,99.218750,4.000000
50%,56999.500000,35.000000,2.129060e+05,0.580000,0.685000,5.000000,-7.004000,1.000000,0.048900,0.169000,0.000042,0.132000,0.464000,122.017000,4.000000
75%,85499.250000,50.000000,2.615060e+05,0.695000,0.854000,8.000000,-5.003000,1.000000,0.084500,0.598000,0.049000,0.273000,0.683000,140.071000,4.000000
max,113999.000000,100.000000,5.237295e+06,0.985000,1.000000,11.000000,4.532000,1.000000,0.965000,0.996000,1.000000,1.000000,0.995000,243.372000,5.000000


In [18]:
# Checking for missing values in the dataset

missing_values = df.isnull().sum()
print(missing_values)

Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64


In [14]:
# Check for duplicates in the dataset
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

Number of duplicate rows: 0


In [19]:
# Check if genre exits in the dataset
# Does the chosen genre exist?
if GENRE in df['track_genre'].unique():
    print(f'The genre "{GENRE}" exists in the dataset.')

The genre "acoustic" exists in the dataset.


In [21]:
# Filter the dataset for the selected genre
df_acoustic = df[df['track_genre'] == GENRE]

df_acoustic

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.03220,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.92400,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.21000,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.90500,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.46900,0.000000,0.0829,0.167,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,1epXnhFfFToKeugQWBCsQg,Harley Poe,Wretched. Filthy. Ugly.,Maria,26,196226,True,0.633,0.8320,...,-6.840,1,0.0863,0.02740,0.000008,0.0957,0.838,130.912,4,acoustic
996,996,5aBpIqXfktr9SSwyAi8Hlj,Taj Mahal,The Essential Taj Mahal,"Corinna - From ""The Natch'l Blues""",29,180920,False,0.621,0.2940,...,-13.468,1,0.1100,0.55800,0.000025,0.0946,0.728,78.599,4,acoustic
997,997,1ZD2MlKRTYKI8oewOt28mc,Pigeon Pit,Treehouse,Tall Cans,26,186461,True,0.648,0.3890,...,-8.275,1,0.0927,0.80700,0.000098,0.1100,0.378,86.360,4,acoustic
998,998,3zVMv0mxjnJRw2BX5ryCqT,Chuck Ragan,Covering Ground,You Get What You Give,26,217293,False,0.303,0.5060,...,-6.890,1,0.0325,0.49600,0.000000,0.1780,0.412,161.463,3,acoustic


In [ ]:
# Observations: (1)
df_acoustic.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1000 non-null   int64  
 1   track_id          1000 non-null   object 
 2   artists           1000 non-null   object 
 3   album_name        1000 non-null   object 
 4   track_name        1000 non-null   object 
 5   popularity        1000 non-null   int64  
 6   duration_ms       1000 non-null   int64  
 7   explicit          1000 non-null   bool   
 8   danceability      1000 non-null   float64
 9   energy            1000 non-null   float64
 10  key               1000 non-null   int64  
 11  loudness          1000 non-null   float64
 12  mode              1000 non-null   int64  
 13  speechiness       1000 non-null   float64
 14  acousticness      1000 non-null   float64
 15  instrumentalness  1000 non-null   float64
 16  liveness          1000 non-null   float64
 17  v

##### 1. Based on the information above, We can see that the dataset has 1000 rows and 21 columns.The columns include various features related to the tracks, such as 'track_name', 'track_artist', 'track_genre', 'track_popularity', and several audio features like 'danceability', 'energy', 'loudness', etc. There are no missing values in the dataset.

In [ ]:
# Observations: (2) and (3)
df_acoustic.describe()

,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,499.500000,42.483000,214896.957000,0.549593,0.435368,5.045000,-9.447843,0.816000,0.043247,0.566816,0.038336,0.153244,0.424023,119.010624,3.885000
std,288.819436,16.541019,55615.279398,0.122203,0.236318,3.588294,3.773481,0.387678,0.025636,0.330443,0.159383,0.105335,0.222394,29.771694,0.426564
min,0.000000,0.000000,59823.000000,0.213000,0.011900,0.000000,-28.276000,0.000000,0.023200,0.000006,0.000000,0.028400,0.041900,53.845000,1.000000
25%,249.750000,29.000000,178664.250000,0.463000,0.254000,2.000000,-11.466000,1.000000,0.030500,0.262250,0.000000,0.100000,0.247750,94.999000,4.000000
50%,499.500000,47.000000,214822.000000,0.558500,0.394500,5.000000,-9.003000,1.000000,0.035200,0.691500,0.000003,0.114500,0.374000,117.751500,4.000000
75%,749.250000,54.000000,246286.500000,0.639000,0.624000,8.000000,-6.637250,1.000000,0.045925,0.861000,0.000213,0.159000,0.569500,139.696750,4.000000
max,999.000000,82.000000,578064.000000,0.879000,0.974000,11.000000,-1.963000,1.000000,0.403000,0.992000,0.978000,0.944000,0.975000,204.961000,5.000000


#### 2. Acoustic tracks are generally quite acoustic: acousticness mean = 0.567, median = 0.692, 75th percentile = 0.861, max = 0.992, This suggests most tracks in this subset are strongly acoustic, not just lightly acoustic

#### 3. They are moderately popular, but not highly popular:            popularity mean = 42.48, median = 47, range = 0 to 82, spread is fairly wide (std ≈ 16.54), Meaning acoustic tracks in this sample are not usually super viral, but they can still reach decent popularity.